# Amazon ML Challenge: data audit and safe preprocessing

## Goal
Explore the three train and three test TSV sources and the training labels without loading the full dataset into memory. Preserve multilingual text. Produce cleaned source TSVs only when you explicitly enable the export cell.

**What this notebook measures by default:** the first `MAX_ROWS_PER_FILE` records of each file. This is a quick diagnostic, **not a random or representative sample**. Set `FULL_SCAN = True` for exact file-wide counts and a bounded reservoir sample across each entire file. The full scan reads roughly 3 GB and will take longer, but keeps memory bounded.

**Challenge rules:** Source 1 is the reference. Sources 2 and 3 contain possible matches. Train covers US and India; test also includes France. Keep country labels open-ended. Ground-truth match IDs are comma-separated *inside a tab-separated file*.

## 1. Setup

Install once if needed: `python -m pip install pandas matplotlib nbformat nbclient ipykernel`. The notebook checks `BER_DATA_ROOT`, the repository's `dataset/` folder, and the standard Kaggle mount. All outputs are bounded previews; source files remain untouched.

In [ ]:
from pathlib import Path
from collections import Counter
import os
import unicodedata

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def locate_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").is_file() and (candidate / "Experiment_Notebooks").is_dir():
            return candidate
    return start

PROJECT_ROOT = locate_project_root(Path.cwd())
data_override = os.environ.get("BER_DATA_ROOT")
data_candidates = [
    Path(data_override).expanduser() if data_override else None,
    PROJECT_ROOT / "dataset",
    PROJECT_ROOT / "student_resource" / "dataset",
    Path("/kaggle/input/datasets/satyapaladugu/amazon-ml-challenge"),
    Path("/kaggle/input/amazon-ml-challenge"),
]
DATA_ROOT = next((path.resolve() for path in data_candidates if path and (path / "train").is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Dataset not found. Set BER_DATA_ROOT to the folder containing train/ and test/.")
FULL_SCAN = True
MAX_ROWS_PER_FILE = 25_000  # used only when FULL_SCAN is False
CHUNK_SIZE = 10_000
PREVIEW_ROWS = 8
RANDOM_SEED = 42
EXPORT_CLEAN_TSV = False  # deliberate opt-in; can write several GB
EXPORT_ROOT = (Path("/kaggle/working") if Path("/kaggle").exists() else PROJECT_ROOT) / "preprocessed"

SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
FILES = {
    "train_source1": DATA_ROOT / "train" / "train_source1.tsv",
    "train_source2": DATA_ROOT / "train" / "train_source2.tsv",
    "train_source3": DATA_ROOT / "train" / "train_source3.tsv",
    "test_source1": DATA_ROOT / "test" / "test_source1.tsv",
    "test_source2": DATA_ROOT / "test" / "test_source2.tsv",
    "test_source3": DATA_ROOT / "test" / "test_source3.tsv",
    "train_ground_truth": DATA_ROOT / "train" / "train_ground_truth.tsv",
}
missing_files = [str(path) for path in FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Missing TSV files:\n" + "\n".join(missing_files))
print(f"Data root: {DATA_ROOT}")
print(f"Scan mode: {'all rows' if FULL_SCAN else f'first {MAX_ROWS_PER_FILE:,} rows per file'}")

## 2. File inventory and schema

File sizes are exact filesystem metadata. Row counts below come from the scan mode chosen above.

In [ ]:
inventory = []
for file_key, path in FILES.items():
    header = pd.read_csv(path, sep="\t", nrows=0, encoding="utf-8").columns.tolist()
    expected = ["source1_entity_id", "matched_entity_ids"] if file_key == "train_ground_truth" else SOURCE_COLUMNS
    if header != expected:
        raise ValueError(f"Unexpected columns in {path.name}: {header}; expected {expected}")
    inventory.append({"file": file_key, "size_MB": round(path.stat().st_size / 1_000_000, 1), "columns": ", ".join(header)})
display(pd.DataFrame(inventory))

## 3. Scan in chunks

Each chunk is parsed as TSV with strings, so IDs keep their exact spelling and empty fields stay empty strings. In quick mode, `nrows` stops parsing after the configured limit. In either mode, a fixed-size reservoir retains example rows from across the scanned portion of each file.

In [ ]:
import random


def script_group(character):
    if not character.isalpha():
        return None
    name = unicodedata.name(character, "")
    for prefix in ("LATIN", "DEVANAGARI", "BENGALI", "ARABIC", "CYRILLIC", "GREEK", "HEBREW", "TAMIL", "TELUGU", "KANNADA", "MALAYALAM", "GUJARATI", "GURMUKHI", "THAI", "HANGUL", "HIRAGANA", "KATAKANA", "CJK"):
        if name.startswith(prefix):
            return prefix
    return "OTHER"


def text_scripts(value):
    return sorted({group for character in value if (group := script_group(character))})


def iter_chunks(path):
    return pd.read_csv(
        path, sep="\t", encoding="utf-8", dtype=str, keep_default_na=False,
        chunksize=CHUNK_SIZE, nrows=None if FULL_SCAN else MAX_ROWS_PER_FILE,
        on_bad_lines="error",
    )


def inspect_file(file_key, path):
    is_truth = file_key == "train_ground_truth"
    counters = {name: Counter() for name in ("country", "name_script", "address_script", "match_count", "match_source")}
    missing = Counter()
    examples = []
    row_count = 0
    rng = random.Random(RANDOM_SEED)
    preview_limit = 1_000
    for chunk in iter_chunks(path):
        for row in chunk.itertuples(index=False, name=None):
            row_count += 1
            record = dict(zip(chunk.columns, row))
            if is_truth:
                matched = [item.strip() for item in record["matched_entity_ids"].split(",") if item.strip()]
                counters["match_count"][len(matched)] += 1
                counters["match_source"].update(item.split("-", 1)[0] for item in matched)
            else:
                for field in SOURCE_COLUMNS:
                    if not record[field].strip():
                        missing[field] += 1
                counters["country"][record["country"].strip() or "<empty>"] += 1
                counters["name_script"].update(text_scripts(record["business_name"]) or ["NONE"])
                counters["address_script"].update(text_scripts(record["business_address"]) or ["NONE"])
            if len(examples) < preview_limit:
                examples.append(record)
            else:
                position = rng.randrange(row_count)
                if position < preview_limit:
                    examples[position] = record
    return {"rows_scanned": row_count, "missing": missing, "counters": counters, "examples": examples}


profiles = {file_key: inspect_file(file_key, path) for file_key, path in FILES.items()}
coverage = "complete" if FULL_SCAN else "early-row sample"
display(pd.DataFrame([
    {"file": key, "rows_scanned": value["rows_scanned"], "coverage": coverage}
    for key, value in profiles.items()
]))

## 4. Inspect raw records

These examples show actual strings before any cleaning. They are a deterministic reservoir sample of the rows scanned. In quick mode this sample is still limited to the first `MAX_ROWS_PER_FILE` rows.

In [ ]:
for file_key in ("train_source1", "train_source2", "train_source3", "test_source1"):
    print(file_key)
    display(pd.DataFrame(profiles[file_key]["examples"]).head(PREVIEW_ROWS))

## 5. Country coverage and missing fields

France appears in test, so any later model must accept unseen country labels. Counts and percentages use the scanned rows as their denominator.

In [ ]:
country_rows = []
missing_rows = []
for file_key, profile in profiles.items():
    if file_key == "train_ground_truth":
        continue
    denominator = profile["rows_scanned"]
    for country, count in profile["counters"]["country"].items():
        country_rows.append({"file": file_key, "country": country, "rows": count, "pct_scanned": round(100 * count / denominator, 2)})
    for field in SOURCE_COLUMNS:
        count = profile["missing"][field]
        missing_rows.append({"file": file_key, "field": field, "empty_rows": count, "pct_scanned": round(100 * count / denominator, 2)})
country_table = pd.DataFrame(country_rows)
missing_table = pd.DataFrame(missing_rows)
display(country_table.sort_values(["file", "rows"], ascending=[True, False]))
display(missing_table)

In [ ]:
pivot = country_table.pivot(index="file", columns="country", values="pct_scanned").fillna(0)
ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 4), width=0.8)
ax.set(title=f"Country share by file ({coverage})", xlabel="Source file", ylabel="Percent of scanned rows")
ax.legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Language script signals

This is **Unicode script detection, not language identification**. Latin can represent English, French, or transliterated Indian names. A record can contribute to more than one script count. Accents and non-Latin writing are preserved.

In [ ]:
script_rows = []
for file_key, profile in profiles.items():
    if file_key == "train_ground_truth":
        continue
    for field, counter_key in (("business_name", "name_script"), ("business_address", "address_script")):
        for script, count in profile["counters"][counter_key].most_common():
            script_rows.append({"file": file_key, "field": field, "script": script, "rows_with_script": count})
script_table = pd.DataFrame(script_rows)
display(script_table.sort_values(["file", "field", "rows_with_script"], ascending=[True, True, False]).head(40))

for file_key in ("train_source2", "test_source1"):
    sample = pd.DataFrame(profiles[file_key]["examples"])
    subset = sample[sample["business_name"].map(lambda value: any(script != "LATIN" for script in text_scripts(value)))]
    print(f"{file_key}: non-Latin name examples in retained sample")
    display(subset[SOURCE_COLUMNS].head(PREVIEW_ROWS))

## 7. Training label shape

One Source 1 record can have zero or many S2/S3 matches. These are training labels only. The test files have no labels.

In [ ]:
truth = profiles["train_ground_truth"]
truth_counts = truth["counters"]["match_count"]
display(pd.DataFrame({"matches_per_source1": list(truth_counts), "rows": list(truth_counts.values())}).sort_values("matches_per_source1").head(25))
print("S2/S3 matched ID occurrences:", dict(truth["counters"]["match_source"]))
print("Singleton share among scanned ground-truth rows:", f"{truth_counts[0] / truth['rows_scanned']:.1%}")
display(pd.DataFrame(truth["examples"]).head(PREVIEW_ROWS))

fig, ax = plt.subplots(figsize=(8, 4))
shown = {min(number, 10): 0 for number in truth_counts}
for number, count in truth_counts.items():
    shown[min(number, 10)] += count
ax.bar([str(number) if number < 10 else "10+" for number in sorted(shown)], [shown[number] for number in sorted(shown)])
ax.set(title=f"Matches per Source 1 label ({coverage})", xlabel="Number of labeled matches", ylabel="Scanned Source 1 rows")
plt.tight_layout()
plt.show()

## 8. Conservative text normalization

`NFKC` harmonizes some Unicode presentation forms; `casefold` makes casing comparable; whitespace is collapsed. This does **not** transliterate scripts, strip accents, remove punctuation, or discard legal suffixes. Keep the raw fields alongside the cleaned fields because normalization can merge distinct spellings. IDs are never normalized.

In [ ]:
def normalize_text(value):
    if not value:
        return ""
    normalized = unicodedata.normalize("NFKC", value)
    return " ".join(normalized.casefold().split())


def clean_source_chunk(chunk):
    cleaned = chunk.copy()
    cleaned["business_name_clean"] = cleaned["business_name"].map(normalize_text)
    cleaned["business_address_clean"] = cleaned["business_address"].map(normalize_text)
    cleaned["country_clean"] = cleaned["country"].map(lambda value: value.strip())
    return cleaned


demo = pd.DataFrame(profiles["train_source2"]["examples"])
display(clean_source_chunk(demo)[["entity_id", "business_name", "business_name_clean", "business_address", "business_address_clean", "country_clean"]].head(PREVIEW_ROWS))
assert normalize_text("  École   राम  ") == "école राम"
assert normalize_text("ＡＢＣ  Ltd") == "abc ltd"


## 9. Optional full-data export

Set `EXPORT_CLEAN_TSV = True` to write six source files in chunks to `EXPORT_ROOT`. The export always reads **all rows**, regardless of scan mode. It does not alter originals or labels. It writes into `.partial` files and renames each only after completion; rerunning overwrites files in the dedicated output folder. Check free disk space first (the new six files may exceed 3 GB).

In [ ]:
def export_clean_sources(output_root):
    output_root.mkdir(parents=True, exist_ok=True)
    summary = []
    for file_key, path in FILES.items():
        if file_key == "train_ground_truth":
            continue
        target = output_root / f"{file_key}_clean.tsv"
        partial = target.with_suffix(".tsv.partial")
        rows_written = 0
        first_chunk = True
        try:
            for chunk in pd.read_csv(path, sep="\t", encoding="utf-8", dtype=str,
                                     keep_default_na=False, chunksize=CHUNK_SIZE, on_bad_lines="error"):
                clean_source_chunk(chunk).to_csv(partial, sep="\t", index=False,
                                                   header=first_chunk, mode="w" if first_chunk else "a",
                                                   encoding="utf-8")
                rows_written += len(chunk)
                first_chunk = False
            partial.replace(target)
        finally:
            if partial.exists():
                partial.unlink()
        summary.append({"file": str(target), "rows_written": rows_written})
    return pd.DataFrame(summary)


if EXPORT_CLEAN_TSV:
    display(export_clean_sources(EXPORT_ROOT))
else:
    print("Export disabled. Set EXPORT_CLEAN_TSV = True in Setup when ready.")

## 10. What to inspect before modeling

1. Compare train and test country and script distributions, especially France in test.
2. Check whether address/name missingness differs by source.
3. Inspect examples of non-Latin names and transliterated records before choosing blocking rules.
4. Measure duplicate IDs and exact cleaned-string collisions on a **full scan** before treating those as global facts.
5. Keep train labels separate from test data; avoid learning match decisions from the unlabeled test set.

The notebook stops at audit and conservative preprocessing. Candidate generation and matching need their own design and validation.